# Tutorial 2: finding subunits with a spike classifier

Tutorial 1 described the cell with an LN model: one linear filter, then one nonlinearity. Here we look *inside* the receptive field for **subunits**, the separately rectified inputs the cell pools, using a small neural network, and test whether a model built from them predicts the frozen-noise response better.

**The idea.** Train a network to answer a yes/no question for every frame: *will the cell spike?* Its input is the stimulus inside the receptive field; it has one hidden layer of rectifying units (ReLU) and a single output, the spike probability. That architecture is itself a subunit model: each hidden unit filters the stimulus spatially and rectifies, and the output combines them. If training works, some hidden units' input weights take the shape of real subunits.

Steps:

1. Train the classifier (`run_subunit_model`) and check that it learned something.
2. Inspect every hidden unit's spatial weights and select subunits with Moran's I.
3. Build a rectified subunit model from them and compare it with the LN model on frozen noise.

This takes a few minutes per run on all 219 trials.

## Setup

In [ ]:
import os
import sys
from pathlib import Path

# Run from the repository root whichever folder Jupyter starts in, so the
# relative paths below and `import src` work.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import numpy as np
from matplotlib import pyplot as plt

from src.analysis.analyze_subunits import get_rf_size, get_subunit_sizes
from src.analysis.model_comparison import correlation_ceiling, prediction_metrics
from src.data_cache import cropped_stimulus, load_spike_data
from src.prediction.sta_model_predictions import (
    binned_response,
    gen_sta_model,
    get_sta_predictions,
    nonlin,
    separable_ln_filter,
)
from src.prediction.subunit_model_predictions import (
    gen_subunit_model,
    get_subunit_predictions,
    subunit_drive,
)
from src.simulation.run_pipeline import run_subunit_model
from src.visualization.plotting_fxns import (
    MODEL_COLORS,
    OKABE_ITO,
    plot_nonlinearity,
    plot_predictions,
    plot_subunits,
    plot_training_curves,
    save_figure,
    use_style,
)

use_style()

In [ ]:
# The data folder can live anywhere: set RGC_DATA_ROOT before starting Jupyter
# (e.g. a share or an external drive); otherwise ./data is used.
DATA_ROOT = Path(os.environ.get("RGC_DATA_ROOT", "data"))
DATA_PATH = DATA_ROOT / "cell_data_01_NC.mat"
STIMULUS_PATH = DATA_ROOT / "stimulus_data"
CACHE_DIR = Path("results/cache")          # disposable: delete it to recompute
STA_PATH = CACHE_DIR / "sta"
FIGURES = Path("results/tutorials/02_classifier_subunit_model")

CELL = 0
MAX_TRIALS = None        # None uses all 219 trials; a small number only checks that the code runs
WIDTH, HEIGHT, REFRESH_RATE = 200, 150, 30   # stimulus pixels and frames per second
DT = 1 / REFRESH_RATE
FILTER_LENGTH = int(0.67 * REFRESH_RATE)     # 20 frames ≈ 0.67 s of stimulus history
SIGMA = 3                                    # ellipse size, in standard deviations

# Classifier settings (the values the original analysis used).
NODE_NUM = 60            # hidden units, i.e. candidate subunits
EPOCHS = 100             # passes through the training data
BATCH_SIZE = 25
LEARNING_RATE = 0.5      # SGD
L1_COEFFICIENT = 1e-4    # pushes weights to zero, keeping maps compact
L2_COEFFICIENT = 0
PATIENCE = None          # None runs every epoch; an integer turns on early stopping
MORANS_THRESHOLD = 0.25
SEED = 0

## 1. Training the classifier

`run_subunit_model` does everything from Tutorial 1 up to the crop (STA, ellipse, cropped stimulus, all cached), then:

- builds one **example** per frame: the 20 frames of stimulus before it, weighted by the temporal STA and averaged over time, giving one value per pixel inside the ellipse; the **label** is 1 if the frame had a spike;
- splits the examples into training, validation and test sets **by whole trials** (25% of trials for testing, 25% of the rest for validation). Neighbouring frames share most of their stimulus history, so splitting frame by frame would leak near-copies of training examples into the test set;
- trains with **balanced sampling**, drawing spike and no-spike frames equally often, because spikes are rare;
- keeps the weights from the epoch with the lowest validation loss.

In [ ]:
result = run_subunit_model(
    CELL, DATA_PATH, STIMULUS_PATH, sta_path=STA_PATH, cache_dir=CACHE_DIR,
    sigma=SIGMA, node_num=NODE_NUM, num_epochs=EPOCHS, batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE, l1_coefficient=L1_COEFFICIENT,
    l2_coefficient=L2_COEFFICIENT, patience=PATIENCE, morans_threshold=MORANS_THRESHOLD,
    random_state=SEED, max_trials=MAX_TRIALS,
)
crop, crop_shape = result.crop, result.ellipse_mask.shape
spikecounts = load_spike_data(DATA_PATH, CACHE_DIR / "spikes", keys=("spk1",))["spk1"][CELL]
num_frames = spikecounts.shape[0]
num_trials = spikecounts.shape[1] if MAX_TRIALS is None else min(MAX_TRIALS, spikecounts.shape[1])
spikecounts = spikecounts[:, :num_trials]
print({name: len(trials) for name, trials in result.split_trials.items()}, "trials per set")

**Did it learn?** Plain accuracy is misleading here: a network that never predicts a spike is right on about 95% of frames. Two measures that are not fooled:

- **Balanced accuracy**: the average of the hit rate on spike frames and on no-spike frames. 0.5 is chance.
- **ROC AUC**: the probability that a random spike frame gets a higher spike probability than a random no-spike frame. 0.5 is chance, 1 is perfect.

The loss curves should fall and level off; a validation loss that rises while training loss keeps falling means overfitting (the restored weights are from the lowest validation point).

In [ ]:
spike_fraction = (spikecounts[FILTER_LENGTH:] > 0).mean()
print(f"test set: accuracy {result.test_accuracy:.3f} (always 'no spike' would score "
      f"{1 - spike_fraction:.3f})")
print(f"          balanced accuracy {result.test_balanced_accuracy:.3f}, ROC AUC {result.test_auc:.3f} "
      f"(0.5 = chance)")

fig, ax = plt.subplots(figsize=(5, 3.2))
plot_training_curves(result.losses, ax=ax, ylabel="Cross-Entropy")
ax.set_title("Classifier Training")
fig.tight_layout()
save_figure(fig, FIGURES / "Classifier Training Curves", caption=(
    "Binary cross-entropy (without the L1 term) on the training and validation trials per "
    f"epoch. Test set: balanced accuracy {result.test_balanced_accuracy:.3f}, ROC AUC "
    f"{result.test_auc:.3f}."))

## Which nodes are subunits?

Every hidden node has one weight per pixel, so each can be drawn as a map on the crop. Some nodes learn smooth, compact blobs: candidate subunits. Others stay noisy. **Moran's I** scores how smooth a map is: how similar each pixel's weight is to its four neighbours'. It is near 0 for salt-and-pepper noise and close to 1 for a smooth blob, and it is computed on the pixels inside the ellipse only. Nodes scoring above `MORANS_THRESHOLD` are kept as subunits.

The maps below are sorted by Moran's I; the title of each is the node number and its score.

In [ ]:
order = np.argsort(np.nan_to_num(result.morans_i, nan=-np.inf))[::-1]
titles = [f"{n}: I={result.morans_i[n]:.2f}" for n in order]
fig, _ = plot_subunits(result.node_weights[order], crop_shape, columns=10, titles=titles,
                       mask=result.ellipse_mask)
fig.suptitle("All Classifier Hidden-Node Weights, Sorted By Moran's I", y=1.0)
fig.tight_layout()
save_figure(fig, FIGURES / "All Node Weights", caption=(
    "Spatial weight map of every node (classifier hidden-node weights) on the cropped window (red positive, blue "
    "negative, white zero; grey outline, ellipse mask), sorted by Moran's I computed inside "
    "the mask."))

fig, ax = plt.subplots(figsize=(4.5, 3))
ax.hist(result.morans_i[np.isfinite(result.morans_i)], bins=20, color="0.6")
ax.axvline(MORANS_THRESHOLD, color=OKABE_ITO["vermillion"], lw=1.5)
ax.set(xlabel="Moran's I", ylabel="Nodes",
       title=f"{len(result.subunits)} Of {len(result.morans_i)} Nodes Selected")
fig.tight_layout()
save_figure(fig, FIGURES / "Morans I Distribution", caption=(
    f"Moran's I of every node; nodes right of the vermillion line (threshold "
    f"{MORANS_THRESHOLD}) are kept as subunits."))

In [ ]:
if len(result.subunits):
    fig, _ = plot_subunits(result.subunits, crop_shape, columns=min(6, len(result.subunits)),
                           mask=result.ellipse_mask)
    fig.suptitle("Selected Subunits", y=1.02)
    fig.tight_layout()
    save_figure(fig, FIGURES / "Selected Subunits", caption=(
        f"The {len(result.subunits)} nodes whose Moran's I exceeds {MORANS_THRESHOLD}."))
    sizes = get_subunit_sizes(result.subunits, *crop_shape)
    print("effective diameter of each positive subunit (µm):", np.round(sizes, 0))
    print(f"receptive field: {get_rf_size(result.spatial_sta[crop]):.0f} µm")

## The LN baseline

To judge the subunit model, fit the LN model as in Tutorial 1, on the same trials and crop, with the **separable** filter: the spatial STA inside the ellipse times the temporal STA. The subunit model has the same structure (spatial maps inside the ellipse, the same temporal STA), so the only difference between the two is the subunits' rectification. A full-STA LN filter would also be noisier, which would make the subunit model look better for the wrong reason.

In [ ]:
stimuli = cropped_stimulus(
    STIMULUS_PATH, crop, WIDTH, HEIGHT, num_frames, num_trials, CACHE_DIR / "stimuli")
ln_filter = separable_ln_filter(result.spatial_sta, result.temporal_sta, crop,
                                result.ellipse_mask)
ln_parameters = gen_sta_model(
    WIDTH, HEIGHT, spikecounts, crop, *crop_shape, FILTER_LENGTH, num_frames, num_trials,
    ln_filter, STIMULUS_PATH, stimuli=stimuli,
)
ln_predictions, _, measured_rate, time = get_sta_predictions(
    CELL, ln_parameters, ln_filter, crop, *crop_shape, DATA_PATH, STIMULUS_PATH,
    filter_length=FILTER_LENGTH, cache_dir=CACHE_DIR,
)
repeats = load_spike_data(DATA_PATH, CACHE_DIR / "spikes", keys=("spk2",))["spk2"][CELL]
ceiling = correlation_ceiling(repeats, FILTER_LENGTH)
ln_metrics = prediction_metrics(measured_rate, ln_predictions, FILTER_LENGTH,
                                ceiling=ceiling.max_correlation)
print(f"separable LN: r = {ln_metrics.correlation:.3f} (noise ceiling {ceiling.max_correlation:.3f})")

## The subunit model

The selected subunits become the spatial filters of a **rectified subunit model**:

1. Weight the recent stimulus by the temporal STA and average over time, giving one value per pixel.
2. Pass that through each subunit's spatial filter and **rectify** (set negative values to zero): each subunit responds only to its preferred contrast.
3. Add the rectified subunit outputs with one weight each, and pass the sum through a softplus, as in the LN model.

The combination weights are chosen so that the weighted subunits add up to the cell's spatial STA (least squares; subunits are first scaled to unit size, and the STA's sign is flipped for OFF cells). The softplus is then fitted to the binned drive, as for the LN model.

`gen_subunit_model` returns the fitted softplus parameters, the combination weights and the **normalized** subunits; pass those normalized subunits on to `get_subunit_predictions`.

In [ ]:
if len(result.subunits) == 0:
    raise RuntimeError(
        "No node passed the Moran's I threshold, so there is no subunit model to build. "
        "Look at the node maps above; train longer or on more trials before changing "
        "MORANS_THRESHOLD.")
subunit_parameters, combination_weights, normalized_subunits = gen_subunit_model(
    result.subunits, WIDTH, HEIGHT, spikecounts, FILTER_LENGTH, num_frames, num_trials,
    result.spatial_sta, result.temporal_sta, crop, *crop_shape, STIMULUS_PATH,
    stimuli=stimuli,
)
print("combination weights:", np.round(combination_weights, 3))

drive = np.concatenate([
    subunit_drive(stimuli[t], normalized_subunits, combination_weights, result.temporal_sta,
                  FILTER_LENGTH)
    for t in range(num_trials)
])
counts = spikecounts[FILTER_LENGTH:, :].T.ravel()
centers, binned_rates = binned_response(drive, counts, DT)

In [ ]:
fig, (ax_weights, ax_curve) = plt.subplots(1, 2, figsize=(9, 3.4))
ax_weights.bar(np.arange(len(combination_weights)), combination_weights,
               color=MODEL_COLORS["Classifier Subunit"])
ax_weights.axhline(0, color="0.6", lw=0.8)
ax_weights.set(xlabel="Subunit", ylabel="Combination Weight", title="Subunit Weights")
plot_nonlinearity(centers, binned_rates, lambda d: nonlin(d, *subunit_parameters),
                  ax=ax_curve, color=MODEL_COLORS["Classifier Subunit"])
ax_curve.set_title("Subunit Model Nonlinearity")
fig.tight_layout()
save_figure(fig, FIGURES / "Subunit Model Fit", caption=(
    "Left: weight of each selected subunit in the subunit model (least-squares fit of the "
    "normalized subunits to the spatial STA). Right: mean firing rate in 40 equal-count bins "
    "of the subunit model's drive (points) and the fitted softplus (line)."))

## Comparing the models on frozen noise

Both models now predict the same held-out frozen-noise frames. A subunit model that captures real nonlinear pooling should correlate better with the measured rate than the LN model, and come closer to the noise ceiling.

In [ ]:
subunit_predictions, _, _, _ = get_subunit_predictions(
    CELL, subunit_parameters, combination_weights, normalized_subunits, result.temporal_sta,
    crop, *crop_shape, DATA_PATH, STIMULUS_PATH, filter_length=FILTER_LENGTH,
    cache_dir=CACHE_DIR,
)
subunit_metrics = prediction_metrics(measured_rate, subunit_predictions, FILTER_LENGTH,
                                     ceiling=ceiling.max_correlation)
print(f"{'Model':<22}{'r':>7}{'normalized r':>14}{'MSE (Hz²)':>12}")
for name, metrics in (("LN", ln_metrics), ("Classifier Subunit", subunit_metrics)):
    print(f"{name:<22}{metrics.correlation:>7.3f}{metrics.normalized_correlation:>14.3f}"
          f"{metrics.mse:>12.1f}")

fig, (ax_trace, ax_bars) = plt.subplots(1, 2, figsize=(12, 3.2),
                                        gridspec_kw={"width_ratios": [4, 1]})
plot_predictions(ln_predictions, measured_rate, time, FILTER_LENGTH, label="LN", ax=ax_trace)
ax_trace.plot(time[FILTER_LENGTH:], subunit_predictions, label="Classifier Subunit", lw=1.2,
              color=MODEL_COLORS["Classifier Subunit"])
ax_trace.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1))
ax_trace.set_title(f"Cell {CELL} Frozen Noise")
names = ["LN", "Classifier Subunit"]
ax_bars.bar([0, 1], [ln_metrics.correlation, subunit_metrics.correlation],
            color=[MODEL_COLORS[n] for n in names])
ax_bars.axhline(ceiling.max_correlation, color="0.4", ls="--", lw=1)
ax_bars.text(1.45, ceiling.max_correlation, "Noise\nCeiling", va="center", fontsize=8)
ax_bars.set(xticks=[0, 1], xticklabels=["LN", "Subunit"], ylabel="Correlation", ylim=(0, 1),
            title="Correlation")
fig.tight_layout()
save_figure(fig, FIGURES / "Frozen Noise Model Comparison", caption=(
    f"Left: measured firing rate of cell {CELL} on the frozen noise, with the LN and "
    f"classifier subunit model predictions. Right: each model's correlation with the measured "
    f"rate; dashed line, noise ceiling (best achievable correlation given the repeats' "
    f"variability). LN r = {ln_metrics.correlation:.3f}, subunit r = "
    f"{subunit_metrics.correlation:.3f}, ceiling {ceiling.max_correlation:.3f}."))

## Things to try

- **Repeat with other seeds** (`SEED`). Networks find somewhat different layouts each run; `src.analysis.model_comparison.select_stable_layout` picks the most typical layout across runs.
- **Vary `NODE_NUM` and `L1_COEFFICIENT`** with `scripts/run_hyperparameter_sweep.py`, which records every run's held-out loss, balanced accuracy and subunit count.
- **Compare with Tutorial 3**, which finds subunits with an autoencoder trained only on spike frames.